# 02 Run Modeling

This notebook is the interactive companion to `scripts/run_modeling.py`.

It uses the same reusable modeling modules to:

- load the canonical processed dataset
- derive an experiment-specific binary target
- split train / validation / test data
- select valid modeling features
- train and compare candidate models
- inspect feature importance and threshold behavior
- optionally save outputs using `wtfd.models.artifacts`

The goal is to keep the notebook focused on experiment inspection and interpretation while sharing as much logic as possible with the script workflow.


In [ ]:
import gc
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from wtfd.models.artifacts import create_run_dir, save_model_outputs
from wtfd.models.experiments import get_experiment_config, list_available_experiments
from wtfd.models.feature_selector import (
    build_feature_matrix,
    summarize_feature_selection,
    validate_no_leakage_columns_in_features,
)
from wtfd.models.metrics import build_threshold_sweep_table
from wtfd.models.model_registry import get_model_config
from wtfd.models.splitter import WindFarmSplitter
from wtfd.models.trainer import WindFaultTrainer
from wtfd.utils.logging_utils import get_logger

logger = get_logger(__name__)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")
logger.info("Initialized 02_run_modeling notebook environment.")


## 1. Configuration

In [ ]:
# Core paths
PROJ_ROOT = Path("..")
MASTER_DATASET_PATH = PROJ_ROOT / "data" / "processed" / "master_dataset.parquet"
OUTPUT_ROOT = PROJ_ROOT / "artifacts" / "modeling"

# Experiment selection
AVAILABLE_EXPERIMENTS = list_available_experiments()
EXPERIMENT_NAME = "pre_72h"

# Optional model override
MODEL_OVERRIDE = None  # Example: ["logistic", "rf"]

# Split / reproducibility settings
TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15
N_SPLITS = 5
RANDOM_STATE = 42

# Feature-selection settings
FEATURE_SUBSET = "all"
NUMERIC_ONLY = True

# Saving settings
SAVE_OUTPUTS = True
RUN_NAME = None  # Example: "pre72h_eventtime_manual"

TOP_N_FEATURES = 20

logger.info(
    "Notebook configuration | experiment=%s | numeric_only=%s | feature_subset=%s | save_outputs=%s",
    EXPERIMENT_NAME,
    NUMERIC_ONLY,
    FEATURE_SUBSET,
    SAVE_OUTPUTS,
)

if EXPERIMENT_NAME not in AVAILABLE_EXPERIMENTS:
    raise ValueError(
        f"Unknown EXPERIMENT_NAME='{EXPERIMENT_NAME}'. "
        f"Available: {AVAILABLE_EXPERIMENTS}"
    )

experiment = get_experiment_config(EXPERIMENT_NAME)
model_names = MODEL_OVERRIDE if MODEL_OVERRIDE is not None else experiment["models"]

config_df = pd.DataFrame(
    [
        {
            "experiment_name": EXPERIMENT_NAME,
            "description": experiment["description"],
            "horizon_hours": experiment["horizon_hours"],
            "include_event": experiment["include_event"],
            "drop_buffer": experiment["drop_buffer"],
            "keep_only_relevant_rows": experiment["keep_only_relevant_rows"],
            "split_method": experiment["split_method"],
            "optimize_for": experiment["optimize_for"],
            "models": ", ".join(model_names),
            "numeric_only": NUMERIC_ONLY,
            "feature_subset": FEATURE_SUBSET,
            "save_outputs": SAVE_OUTPUTS,
        }
    ]
)
display(config_df)


## 2. Load canonical processed dataset and derive the experiment target

In [ ]:
if not MASTER_DATASET_PATH.exists():
    logger.error("Master dataset not found: %s", MASTER_DATASET_PATH)
    raise FileNotFoundError(f"Master dataset not found: {MASTER_DATASET_PATH}")

logger.info("Loading canonical dataset from %s", MASTER_DATASET_PATH)
df = pd.read_parquet(MASTER_DATASET_PATH)
logger.info("Loaded canonical dataset with shape=%s", df.shape)

splitter = WindFarmSplitter(n_splits=N_SPLITS, random_state=RANDOM_STATE)

modeling_df = splitter.create_binary_target_from_state(
    df=df,
    horizon_hours=experiment["horizon_hours"],
    include_event=experiment["include_event"],
    drop_buffer=experiment["drop_buffer"],
    keep_only_relevant_rows=experiment["keep_only_relevant_rows"],
    target_col="target",
)

logger.info(
    "Constructed modeling dataset for experiment=%s with shape=%s and positive_rate=%.6f",
    EXPERIMENT_NAME,
    modeling_df.shape,
    float(modeling_df["target"].mean()),
)

dataset_summary_df = pd.DataFrame(
    [
        {
            "rows": len(modeling_df),
            "columns": modeling_df.shape[1],
            "unique_farms": modeling_df["farm_id"].nunique() if "farm_id" in modeling_df.columns else pd.NA,
            "unique_turbines": modeling_df["asset_id"].nunique() if "asset_id" in modeling_df.columns else pd.NA,
            "unique_events": modeling_df["event_id"].nunique() if "event_id" in modeling_df.columns else pd.NA,
            "positive_rate": modeling_df["target"].mean(),
        }
    ]
)
display(dataset_summary_df)

if "state_name" in modeling_df.columns:
    state_counts = (
        modeling_df["state_name"]
        .value_counts(dropna=False)
        .rename_axis("state_name")
        .reset_index(name="count")
    )
    display(state_counts)

display(modeling_df.head())


## 3. Create train / validation / test splits

In [ ]:
split_method = experiment["split_method"]
logger.info(
    "Creating split using split_method=%s | train=%.2f | val=%.2f | test=%.2f",
    split_method,
    TRAIN_SIZE,
    VAL_SIZE,
    TEST_SIZE,
)

if split_method == "event_time":
    train_df, val_df, test_df = splitter.get_event_level_time_split(
        modeling_df,
        train_size=TRAIN_SIZE,
        val_size=VAL_SIZE,
        test_size=TEST_SIZE,
    )
elif split_method == "group_time":
    train_df, val_df, test_df = splitter.get_grouped_time_split_by_turbine(
        modeling_df,
        train_size=TRAIN_SIZE,
        val_size=VAL_SIZE,
        test_size=TEST_SIZE,
    )
elif split_method == "global_time":
    train_df, val_df, test_df = splitter.get_train_val_test_split_by_time(
        modeling_df,
        train_size=TRAIN_SIZE,
        val_size=VAL_SIZE,
        test_size=TEST_SIZE,
    )
else:
    raise ValueError(f"Unsupported split_method: {split_method}")

split_sizes_df = pd.DataFrame(
    [
        {"split": "train", "rows": len(train_df), "positive_rate": train_df['target'].mean()},
        {"split": "val", "rows": len(val_df), "positive_rate": val_df['target'].mean()},
        {"split": "test", "rows": len(test_df), "positive_rate": test_df['target'].mean()},
    ]
)
display(split_sizes_df)


## 4. Select valid model features

In [ ]:
feature_summary = summarize_feature_selection(
    train_df,
    numeric_only=NUMERIC_ONLY,
    feature_subset=FEATURE_SUBSET,
)
feature_names = feature_summary["selected_features"]
validate_no_leakage_columns_in_features(feature_names)

display(
    pd.DataFrame(
        [
            {
                "n_total_columns": feature_summary["n_total_columns"],
                "n_selected_features": feature_summary["n_selected_features"],
                "numeric_only": feature_summary["numeric_only"],
                "feature_subset": feature_summary["feature_subset"],
            }
        ]
    )
)
display(pd.Series(feature_names[:50], name="selected_feature").to_frame())

X_train = build_feature_matrix(
    train_df,
    numeric_only=NUMERIC_ONLY,
    feature_subset=FEATURE_SUBSET,
)
y_train = train_df["target"].copy()

X_val = build_feature_matrix(
    val_df,
    numeric_only=NUMERIC_ONLY,
    feature_subset=FEATURE_SUBSET,
)
y_val = val_df["target"].copy()

X_test = build_feature_matrix(
    test_df,
    numeric_only=NUMERIC_ONLY,
    feature_subset=FEATURE_SUBSET,
)
y_test = test_df["target"].copy()

logger.info(
    "Prepared X/y matrices | X_train=%s | X_val=%s | X_test=%s",
    X_train.shape,
    X_val.shape,
    X_test.shape,
)


## 5. Model tournament

In [ ]:
tournament_rows = []
detailed_results = {}

for model_name in model_names:
    model_config = get_model_config(model_name)

    logger.info("Training candidate model: %s", model_name)
    trainer = WindFaultTrainer(
        model_type=model_config["model_type"],
        params=model_config["params"],
        random_state=RANDOM_STATE,
    )

    tuned_threshold = trainer.fit_and_tune(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        optimize_for=experiment["optimize_for"],
    )

    train_metrics = trainer.evaluate_detailed(X_train, y_train)
    val_metrics = trainer.evaluate_detailed(X_val, y_val)
    test_metrics = trainer.evaluate_detailed(X_test, y_test)

    tournament_rows.append(
        {
            "model_name": model_name,
            "model_type": model_config["model_type"],
            "threshold": tuned_threshold,
            "train_precision": train_metrics["precision"],
            "train_recall": train_metrics["recall"],
            "train_f1": train_metrics["f1"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
            "test_precision": test_metrics["precision"],
            "test_recall": test_metrics["recall"],
            "test_f1": test_metrics["f1"],
            "test_roc_auc": test_metrics["roc_auc"],
            "test_pr_auc": test_metrics["pr_auc"],
            "test_balanced_accuracy": test_metrics["balanced_accuracy"],
            "test_specificity": test_metrics["specificity"],
        }
    )

    detailed_results[model_name] = {
        "trainer": trainer,
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "model_config": model_config,
    }

    gc.collect()

results_df = (
    pd.DataFrame(tournament_rows)
    .sort_values(
        by=["test_f1", "test_recall", "test_precision"],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)
display(results_df)


## 6. Inspect the winning model

In [ ]:
best_model_name = results_df.iloc[0]["model_name"]
best_bundle = detailed_results[best_model_name]
best_trainer = best_bundle["trainer"]
best_test_metrics = best_bundle["test_metrics"]

logger.info(
    "Winning model=%s | threshold=%.6f | test_f1=%.6f",
    best_model_name,
    best_trainer.best_threshold,
    best_test_metrics["f1"],
)

metric_snapshot_df = pd.DataFrame(
    [
        {
            "model_name": best_model_name,
            "threshold": best_trainer.best_threshold,
            "precision": best_test_metrics["precision"],
            "recall": best_test_metrics["recall"],
            "f1": best_test_metrics["f1"],
            "roc_auc": best_test_metrics["roc_auc"],
            "pr_auc": best_test_metrics["pr_auc"],
            "balanced_accuracy": best_test_metrics["balanced_accuracy"],
            "specificity": best_test_metrics["specificity"],
        }
    ]
)
display(metric_snapshot_df)


## 7. Feature importance / coefficient inspection

In [ ]:
importance_df = best_trainer.get_feature_importance(feature_names).copy()
display(importance_df.head(TOP_N_FEATURES))

plot_df = importance_df.head(TOP_N_FEATURES).iloc[::-1].copy()
value_col = "importance" if "importance" in plot_df.columns else "coefficient"
x_label = "Importance" if value_col == "importance" else "Coefficient"

plt.figure(figsize=(10, 6))
plt.barh(plot_df["feature"], plot_df[value_col])
plt.title(f"Top {TOP_N_FEATURES} Features - {best_model_name}")
plt.xlabel(x_label)
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## 8. Threshold sweep diagnostics

In [ ]:
best_test_probs = best_trainer.predict_proba(X_test)
threshold_df = build_threshold_sweep_table(
    y_true=y_test,
    y_prob=best_test_probs,
)

display(
    threshold_df.sort_values(
        by=["f1", "recall", "precision"],
        ascending=[False, False, False],
    ).head(15)
)

plt.figure(figsize=(10, 6))
plt.plot(threshold_df["threshold"], threshold_df["precision"], label="Precision")
plt.plot(threshold_df["threshold"], threshold_df["recall"], label="Recall")
plt.plot(threshold_df["threshold"], threshold_df["f1"], label="F1")
plt.axvline(best_trainer.best_threshold, linestyle="--", label="Tuned threshold")
plt.title(f"Threshold Sweep - {best_model_name}")
plt.xlabel("Threshold")
plt.ylabel("Metric value")
plt.legend()
plt.tight_layout()
plt.show()


## 9. Optional artifact saving

In [ ]:
run_dir = None

if SAVE_OUTPUTS:
    run_dir = create_run_dir(
        output_root=OUTPUT_ROOT,
        experiment_name=EXPERIMENT_NAME,
        run_name=RUN_NAME,
    )

    metadata = {
        "experiment_name": EXPERIMENT_NAME,
        "experiment_config": experiment,
        "data_path": str(MASTER_DATASET_PATH),
        "feature_selection": feature_summary,
        "split_sizes": {
            "train": len(train_df),
            "val": len(val_df),
            "test": len(test_df),
        },
        "best_model_name": best_model_name,
        "best_threshold": float(best_trainer.best_threshold),
        "best_test_metrics": {
            "precision": float(best_test_metrics["precision"]),
            "recall": float(best_test_metrics["recall"]),
            "f1": float(best_test_metrics["f1"]),
            "roc_auc": float(best_test_metrics["roc_auc"]),
            "pr_auc": float(best_test_metrics["pr_auc"]),
        },
    }

    saved_paths = save_model_outputs(
        run_dir=run_dir,
        results_df=results_df,
        feature_importance_df=importance_df,
        threshold_df=threshold_df,
        metadata=metadata,
    )

    logger.info("Saved notebook artifacts to %s", run_dir)
    display(pd.DataFrame([{"artifact": k, "path": str(v)} for k, v in saved_paths.items()]))
else:
    logger.info("SAVE_OUTPUTS=False; no artifacts were written.")
    print("Artifact saving skipped because SAVE_OUTPUTS=False.")


## 10. Optional next steps

Good follow-on experiments for this notebook:

- compare `pre_24h`, `pre_48h`, and `pre_72h`
- compare `include_event=False` vs `include_event=True` in `experiments.py`
- compare `event_time` against `group_time`
- add experiment-specific model overrides
- load prior run outputs from `artifacts/modeling/` for cross-run comparison
